<a href="https://colab.research.google.com/github/GodishalaAshwith/DeepLearning/blob/main/DLExternal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 16
# Implement CNN on MNITST Dataset

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.datasets import mnist

# Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize and reshape
X_train = X_train.reshape(-1,28,28,1) / 255.0
X_test = X_test.reshape(-1,28,28,1) / 255.0

# Build CNN
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),   # prevents overfitting
    Dense(10, activation='softmax')
])

# Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train
history = model.fit(X_train, y_train,
                    epochs=5)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

In [ ]:

#2. Implement the MLP using the Types of GD (BGD,SGD,Mini BatchGD, SGD with Momentum, SGD with Nesterov,Adagrad, RMSProp,Adadelta and Adam)
# for learning XOR operation.

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta

# XOR dataset
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([[0],[1],[1],[0]])

batches = {"BGD":4, "SGD":1, "MiniBatch":2}

for gd, batch in batches.items():
    print("\n", gd)

    optimizers = {
        "SGD": SGD(learning_rate=0.1),
        "Momentum": SGD(learning_rate=0.1, momentum=0.9),
        "Nesterov": SGD(learning_rate=0.1, momentum=0.9, nesterov=True),
        "Adagrad": Adagrad(),
        "RMSProp": RMSprop(),
        "Adadelta": Adadelta(),
        "Adam": Adam()
    }

    for name, opt in optimizers.items():

        model = Sequential([
            Input(shape=(2,)),
            Dense(4, activation='relu'),
            Dense(1, activation='sigmoid')
        ])

        model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])

        model.fit(X, y, epochs=500, batch_size=batch, verbose=0)

        loss, acc = model.evaluate(X, y, verbose=0)

        print(name, "Accuracy:", acc)

In [ ]:
#3 IMPLEMENT A SIMPLE PERCEPTRON (Coding a Neuron) and explain the meaning of feed-forward, step, and sigmoid functions

import numpy as np

def sigmoid(x):
  return 1 / (1 + np.exp(-x))

def step(z):
    return 1 if z>=0 else 0

class Neuron:
  def __init__(self, weights, bias):
    self.weights = weights
    self.bias = bias

  def feedforward(self, inputs):
    total = np.dot(self.weights, inputs) + self.bias
    return sigmoid(total)

weights = np.array([0, 1])
bias = 4

n = Neuron(weights, bias)

x = np.array([2, 3])
print(n.feedforward(x))


In [ ]:
#4 Implement AND and OR logic operations using a single perceptron, and verify the correctness of the output using appropriate truth tables. (linear Data)
import numpy as np

def step(x):
  return 1 if x>=0 else 0

class Perceptron:
  def __init__(self,weights,bias):
    self.weights = weights
    self.bias = bias

  def predict(self,input):
    total = np.dot(self.weights,input) + self.bias
    return step(total)

weights = np.array([1,1])
and_gate = Perceptron(weights,-1.5)
or_gate = Perceptron(weights, -0.5)

for x in [(0,0),(0,1),(1,0),(1,1)]:
  print(x, 'And ->' , and_gate.predict(np.array(x)))
print()
for x in [(0,0),(0,1),(1,0),(1,1)]:
  print(x, "Or ->", or_gate.predict(np.array(x)))



In [ ]:
#5.  Implement an MLP using the Gradient Descent algorithm, and analyze the convergence behavior and performance of the network.
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load data
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42)

# MLP using Gradient Descent
model = MLPClassifier(hidden_layer_sizes=(5,),
                      solver='sgd',              # Gradient Descent
                      learning_rate_init=0.0001,
                      max_iter=500,
                      random_state=42)

model.fit(X_train, y_train)

# Predictions
pred = model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, pred))

# Convergence (Loss Curve)
plt.plot(model.loss_curve_)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Convergence of MLP (Gradient Descent)")
plt.show()

In [ ]:
#6. Implement the MLP using the Types of Regularization Techniques. Adding noise to the inputs and outputs, Early stopping, Ensemble methods, Dropouts
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load MNIST Dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Dataset Augmentation + Adding Noise
x_train = x_train + 0.1 * np.random.random(x_train.shape)

# Flatten Images
x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)

# One Hot Encoding
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# MLP Model
model = Sequential([

    Flatten(input_shape=(784,)),

    # L2 Regularization + Parameter Sharing/Tying
    Dense(128,
          activation='relu',
          kernel_regularizer=l2(0.001)),

    # Dropout
    Dropout(0.2),

    Dense(10, activation='softmax')
])

# Compile Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Early Stopping
early = EarlyStopping(monitor='val_loss', patience=3)

# Train Model
model.fit(x_train, y_train,
          epochs=3,
          validation_split=0.2,
          callbacks=[early],
          verbose=1)

# Evaluate Model
loss, acc = model.evaluate(x_test, y_test)

print("Accuracy:", acc)

# Ensemble Method
print("\nEnsemble Method")

models = []

for i in range(3):

    m = Sequential([
        Dense(64, activation='relu', input_shape=(784,)),
        Dense(10, activation='softmax')
    ])

    m.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

    m.fit(x_train, y_train, epochs=3, verbose=0)

    models.append(m)

# Ensemble Prediction
pred = sum([m.predict(x_test) for m in models]) / 3

# Accuracy
acc = np.mean(np.argmax(pred, axis=1) == np.argmax(y_test, axis=1))

print("Ensemble Accuracy:", acc)

In [ ]:
#7 Examine the feasibility of implementing the XOR and XNOR (¬XOR) operations (Non linear data) using a single perceptron.
import numpy as np

def step(x):
    return 1 if x >= 0 else 0

class Perceptron:
    def __init__(self, weights, bias):
        self.weights = np.array(weights)
        self.bias = bias

    def predict(self, x):
        return step(np.dot(self.weights, x) + self.bias)

# Input data
X = np.array([[0,0],[0,1],[1,0],[1,1]])

# XOR and XNOR outputs
Y_xor = np.array([0,1,1,0])
Y_xnor = np.array([1,0,0,1])

# Try a perceptron
p = Perceptron(weights=[1, 1], bias=-1)

print("XOR")
for x, y in zip(X, Y_xor):
    print(x, "->", p.predict(x), "Actual:", y)

print("\nXNOR")
for x, y in zip(X, Y_xnor):
    print(x, "->", p.predict(x), "Actual:", y)

In [ ]:
# 8 Perceptron Learning Algorithm - Movie Like/Dislike Prediction

import numpy as np
import pandas as pd

# --------------------------------------------------
# Create Small Dataset (can also save as CSV)
# f1 = Matt Damon
# f2 = Thriller Genre
# f3 = Christopher Nolan
# f4 = IMDb Rating (0 to 1)
# Output = Like(1) / Dislike(0)
# --------------------------------------------------

data = {
    "f1":[1,0,1,0,1],
    "f2":[1,1,0,0,1],
    "f3":[1,0,1,0,0],
    "f4":[0.9,0.7,0.8,0.3,0.6],
    "like":[1,1,1,0,0]
}

df = pd.DataFrame(data)

X = df[["f1","f2","f3","f4"]].values
Y = df["like"].values

# --------------------------------------------------
# i) MP Perceptron (without weights and bias)
# --------------------------------------------------

print("MP Perceptron")

for x in X:

    s = sum(x)

    if s >= 2:
        pred = 1
    else:
        pred = 0

    print(x, "->", pred)

# --------------------------------------------------
# ii) Perceptron (with weights)
# --------------------------------------------------

print("\nPerceptron with Weights")

w = np.zeros(4)
lr = 0.1

for epoch in range(10):

    for x, y in zip(X, Y):

        pred = 1 if np.dot(w, x) >= 0 else 0

        w = w + lr * (y - pred) * x

print("Weights:", w)

# --------------------------------------------------
# iii) Perceptron (with weights and bias)
# --------------------------------------------------

print("\nPerceptron with Weights and Bias")

w = np.zeros(4)
b = 0

for epoch in range(10):

    for x, y in zip(X, Y):

        pred = 1 if np.dot(w, x) + b >= 0 else 0

        w = w + lr * (y - pred) * x
        b = b + lr * (y - pred)

print("Weights:", w)
print("Bias:", b)

# --------------------------------------------------
# Test Sample Record
# --------------------------------------------------

test = np.array([1,1,1,0.95])

pred = 1 if np.dot(w, test) + b >= 0 else 0

print("\nTest Record:", test)

if pred == 1:
    print("User will LIKE the movie")
else:
    print("User will DISLIKE the movie")

In [ ]:
# 9. Demonstrate that the thresholding logic used by perceptron is very harsh

def perceptron(x, w, b):

    s = x*w + b

    # Hard Threshold
    if s >= 0:
        return 1
    else:
        return 0

# Inputs very close to threshold
inputs = [-0.1, -0.01, 0, 0.01, 0.1]

w = 1
b = 0

print("Input  Output")

for x in inputs:
    print(x, "\t", perceptron(x, w, b))

print("\nObservation:")
print("Very small change near threshold changes output suddenly.")
print("Hence perceptron thresholding is harsh.")

In [ ]:
# 10.  Implement an MLP by varying bias, weights, and learning rate, and record observations for different learning rate values. Plot a graph showing the relationship between loss (error) and learning rate.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

# Dataset
X, y = make_classification(
    n_samples=500,
    n_features=10,
    random_state=42
)

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# Different Learning Rates
learning_rates = [0.0001, 0.001, 0.01, 0.1]

loss_values = []

print("MLP Observations\n")

for i, lr in enumerate(learning_rates):

    # MLP Model
    model = MLPClassifier(
        hidden_layer_sizes=(5,),
        learning_rate_init=lr,
        max_iter=500,
        random_state=i+1      # changes weights and bias
    )

    # Train Model
    model.fit(X_train, y_train)

    # Predict Probabilities
    y_pred = model.predict_proba(X_test)

    # Calculate Loss
    loss = log_loss(y_test, y_pred)

    loss_values.append(loss)

    # Display Observations
    print("Learning Rate:", lr)

    print("Sample Weight:",
          model.coefs_[0][0][0])

    print("Sample Bias:",
          model.intercepts_[0][0])

    print("Loss:", loss)

    print()

# Graph
plt.plot(learning_rates,
         loss_values,
         marker='o')

plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.title("Loss vs Learning Rate")

plt.grid(True)

plt.show()

In [ ]:
# 11 LeNet

# LeNet Architecture for Digit Classification using MNIST

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D
from tensorflow.keras.layers import Flatten, Dense

# Load MNIST Dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Reshape and Normalize
x_train = x_train.reshape(60000, 28, 28, 1) / 255.0
x_test = x_test.reshape(10000, 28, 28, 1) / 255.0

# LeNet Model
model = Sequential([

    # C1 Convolution Layer
    Conv2D(6, (5,5),
           activation='tanh',
           input_shape=(28,28,1)),

    # S2 Pooling Layer
    AveragePooling2D(2),

    # C3 Convolution Layer
    Conv2D(16, (5,5),
           activation='tanh'),

    # S4 Pooling Layer
    AveragePooling2D(2),

    # Flatten Layer
    Flatten(),

    # Fully Connected Layers
    Dense(120, activation='tanh'),
    Dense(84, activation='tanh'),

    # Output Layer
    Dense(10, activation='softmax')
])

# Compile Model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train Model
model.fit(x_train, y_train,
          epochs=5,
          batch_size=32)

# Test Accuracy
loss, acc = model.evaluate(x_test, y_test)

print("Accuracy:", acc)

In [ ]:
# 12  Implement a Denoising Autoencoder and test it on noisy images.

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D
import numpy as np
import matplotlib.pyplot as plt

# Load Dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize and Reshape
x_train = x_train.reshape(-1,28,28,1) / 255.0
x_test = x_test.reshape(-1,28,28,1) / 255.0

# Add Noise
x_train_noisy = x_train + 0.5 * np.random.normal(size=x_train.shape)
x_test_noisy = x_test + 0.5 * np.random.normal(size=x_test.shape)

# Autoencoder Model
model = Sequential([

    Conv2D(32, 3, activation='relu', padding='same',
           input_shape=(28,28,1)),

    MaxPooling2D(2, padding='same'),

    Conv2D(32, 3, activation='relu', padding='same'),

    UpSampling2D(2),

    Conv2D(1, 3, activation='sigmoid', padding='same')
])

# Compile
model.compile(optimizer='adam', loss='binary_crossentropy')

# Train
model.fit(x_train_noisy, x_train,
          epochs=3,
          batch_size=128,
          verbose=1)

# Predict
decoded = model.predict(x_test_noisy)

# Show Images
for i in range(3):

    plt.subplot(2,3,i+1)
    plt.imshow(x_test_noisy[i].reshape(28,28), cmap='gray')

    plt.subplot(2,3,i+4)
    plt.imshow(decoded[i].reshape(28,28), cmap='gray')

plt.show()

In [ ]:
# 13  Implement AlexNet and explain its key layers.
# AlexNet for MNIST Classification

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout

# Load Dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize and Reshape
x_train = x_train.reshape(-1,28,28,1) / 255.0
x_test = x_test.reshape(-1,28,28,1) / 255.0

# AlexNet Model
model = Sequential([

    # 1st Convolution Layer
    Conv2D(96, 3, activation='relu',
           input_shape=(28,28,1)),

    MaxPooling2D(2),

    # 2nd Convolution Layer
    Conv2D(256, 3, activation='relu'),

    MaxPooling2D(2),

    # 3rd, 4th, 5th Convolution Layers
    Conv2D(384, 3, activation='relu', padding='same'),
    Conv2D(384, 3, activation='relu', padding='same'),
    Conv2D(256, 3, activation='relu', padding='same'),

    MaxPooling2D(2),

    # Fully Connected Layers
    Flatten(),

    Dense(1024, activation='relu'),
    Dropout(0.5),

    Dense(512, activation='relu'),
    Dropout(0.5),

    # Output Layer
    Dense(10, activation='softmax')
])

# Compile Model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train Model
model.fit(x_train, y_train,
          epochs=3,
          batch_size=128)

# Test Accuracy
loss, acc = model.evaluate(x_test, y_test)

print("Accuracy:", acc)

In [ ]:
# 14. Implement RNN for predicting the next character in a sequence

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

X, y = np.array([[[1,0], [0,1]]]), np.array([[1,0]])

model = Sequential([
    SimpleRNN(8, input_shape=(2, 2)),
    Dense(2, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.fit(X, y, epochs=10, verbose=0)

vocab = {0: 'h', 1: 'i'}
pred = np.argmax(model.predict(X, verbose=0))
print(f"Input: 'hi' -> Predicted: '{vocab[pred]}'")

In [ ]:
#15. Undercomplete and Overcomplete Autoencoder

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
import matplotlib.pyplot as plt

# Load Dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Flatten Images
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Undercomplete Autoencoder
under = Sequential([
    Dense(64, activation='relu', input_shape=(784,)),
    Dense(784, activation='sigmoid')
])

under.compile(optimizer='adam', loss='binary_crossentropy')

under.fit(x_train, x_train,
          epochs=3,
          batch_size=128,
          verbose=0)

# Overcomplete Autoencoder
over = Sequential([
    Dense(1024, activation='relu', input_shape=(784,)),
    Dense(784, activation='sigmoid')
])

over.compile(optimizer='adam', loss='binary_crossentropy')

over.fit(x_train, x_train,
         epochs=3,
         batch_size=128,
         verbose=0)

# Reconstruction Loss
print("Undercomplete Loss:",
      under.evaluate(x_test, x_test))

print("Overcomplete Loss:",
      over.evaluate(x_test, x_test))

# Reconstructed Images
u = under.predict(x_test)
o = over.predict(x_test)

# Display Images
for i in range(3):

    plt.subplot(3,3,i+1)
    plt.imshow(x_test[i].reshape(28,28))

    plt.subplot(3,3,i+4)
    plt.imshow(u[i].reshape(28,28))

    plt.subplot(3,3,i+7)
    plt.imshow(o[i].reshape(28,28))

plt.show()

In [ ]:
#16.  Implement BERT model for text classification.
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="bert-base-uncased"
)

result = classifier("I love this movie")

if result[0]['label'] == 'LABEL_1':
    print("Positive")
else:
    print("Negative")

In [ ]:
#17 Simple GAN on MNIST

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np
import matplotlib.pyplot as plt

# Load Dataset
(x_train, _), (_, _) = mnist.load_data()

# Normalize
x_train = x_train.reshape(-1,784) / 255.0

# Generator
generator = Sequential([
    Dense(128, activation='relu', input_shape=(100,)),
    Dense(784, activation='sigmoid')
])

# Discriminator
discriminator = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dense(1, activation='sigmoid')
])

discriminator.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

# GAN Model
discriminator.trainable = False

gan = Sequential([generator, discriminator])

gan.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

# Training
for i in range(100000):

    # Real Images
    idx = np.random.randint(0, x_train.shape[0], 32)

    real = x_train[idx]

    # Fake Images
    noise = np.random.normal(0,1,(32,100))

    fake = generator.predict(noise, verbose=0)

    # Train Discriminator
    discriminator.train_on_batch(real,
                                 np.ones((32,1)))

    discriminator.train_on_batch(fake,
                                 np.zeros((32,1)))

    # Train Generator
    gan.train_on_batch(noise,
                       np.ones((32,1)))

# Generate Image
noise = np.random.normal(0,1,(1,100))

img = generator.predict(noise, verbose=0)

# Display Generated Image
plt.imshow(img.reshape(28,28), cmap='gray')

plt.show()

In [ ]:
#18. LSTM and GRU on MNIST Dataset

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense

# Load Dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# ---------------- LSTM ----------------

lstm = Sequential([
    LSTM(64, input_shape=(28,28)),
    Dense(10, activation='softmax')
])

lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm.fit(x_train, y_train,
         epochs=1,
         batch_size=128)

loss, acc = lstm.evaluate(x_test, y_test)

print("LSTM Accuracy:", acc)

# ---------------- GRU ----------------

gru = Sequential([
    GRU(64, input_shape=(28,28)),
    Dense(10, activation='softmax')
])

gru.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

gru.fit(x_train, y_train,
        epochs=1,
        batch_size=128)

loss, acc = gru.evaluate(x_test, y_test)

print("GRU Accuracy:", acc)

In [ ]:
# 19  Implement Sparse and contractive Autoencoder.

# Sparse and Contractive Autoencoder

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import regularizers

# Load Dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize and Flatten
x_train = x_train.reshape(-1,784) / 255.0
x_test = x_test.reshape(-1,784) / 255.0

# ---------------- Sparse Autoencoder ----------------

sparse = Sequential([

    Dense(64,
          activation='relu',
          activity_regularizer=regularizers.l1(1e-4),
          input_shape=(784,)),

    Dense(784, activation='sigmoid')
])

sparse.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

sparse.fit(x_train, x_train,
           epochs=3,
           batch_size=128,
           verbose=1)

print("Sparse AE Loss:",
      sparse.evaluate(x_test, x_test))

# ---------------- Contractive Autoencoder ----------------

contractive = Sequential([

    Dense(64,
          activation='relu',
          kernel_regularizer=regularizers.l2(1e-4),
          input_shape=(784,)),

    Dense(784, activation='sigmoid')
])

contractive.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

contractive.fit(x_train, x_train,
                epochs=3,
                batch_size=128,
                verbose=1)

print("Contractive AE Loss:",
      contractive.evaluate(x_test, x_test))

In [ ]:
# 20.  Implement Transformer model

from transformers import pipeline

# Transformer Model
model = pipeline(
    "text-classification",
    model="distilbert-base-uncased"
)

# Prediction
result = model("Transformers are powerful models")

print(result)